In [33]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
import geopandas as gpd
from shapely.geometry import Point
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az

import sys
import botorch
import os
import time
from contextlib import ExitStack

import gpytorch
import gpytorch.settings as gpts
import torch
from gpytorch.constraints import Interval
from gpytorch.distributions import MultivariateNormal
from gpytorch.kernels import RBFKernel, RFFKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood
from torch.quasirandom import SobolEngine
from torch import Tensor

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from botorch.fit import fit_gpytorch_mll
from botorch.generation import MaxPosteriorSampling
from botorch.models import SingleTaskGP
from botorch.test_functions import Hartmann
from botorch.utils.sampling import draw_sobol_samples
from botorch.sampling.pathwise.posterior_samplers import get_matheron_path_model

jobs_data_t0 = pd.read_csv('data/jobs/jobs_by_census_tract_2010_with_coordinates.csv')
building_permits_data_t0 = pd.read_csv('data/building_permits/Building_Permits_t0.csv')

jobs_data_t1 = pd.read_csv('data/jobs/jobs_by_census_tract_2020_with_coordinates.csv')
building_permits_data_t1 = pd.read_csv('data/building_permits/Building_Permits_t1.csv')

tracts = gpd.read_file('data/tl_2020_17_tract/tl_2020_17_tract.shp')

training_data = pd.read_csv('training_data_t0-t1.csv')
# remove tracts with missing gentrification scores
training_data = training_data[~training_data['gentrification_score'].isna()]
# mark the top 10% of census tracts by gentrification score as gentrifying (label = 1). The rest are non-gentrifying (label = 0).
training_data['gentrifying'] = (training_data['gentrification_score'] >= training_data['gentrification_score'].quantile(0.9)).astype(int)


def census_tract_to_coordinates(geoid):
    """Given a Census tract or block GEOID, return its coordinates."""
    if (len(str(geoid)) == 11):
        tract = tracts[tracts['GEOID'] == str(geoid)]
    # elif (len(str(geoid)) == 15):
    #     tract = blocks[blocks['GEOID10'] == str(geoid)]
    if tract.empty:
        return pd.NA, pd.NA
    lat = tract.geometry.iloc[0].centroid.y
    lon = tract.geometry.iloc[0].centroid.x
    return lat, lon

def calculate_distances(lat, lon, df, lat_col='latitude', lon_col='longitude'):
    """Vectorized Euclidean distance in degrees from one point to all rows in df."""
    dlat = df[lat_col].to_numpy(dtype=np.float64) - lat
    dlon = df[lon_col].to_numpy(dtype=np.float64) - lon
    return np.hypot(dlat, dlon)

def calculate_gravity_score(df, distance, alpha, year_var=None, min_year=None):
    distances = df['distance'].to_numpy(dtype=np.float64)
    mask = distances < distance.item()
    if year_var is not None:
        mask &= df[year_var].astype(int).to_numpy() >= min_year

    if 'C000' in df:
        weights = df['C000'].fillna(1).to_numpy(dtype=np.float64)[mask]
    else:
        weights = 1.0
    return np.sum(weights / (distances[mask] ** alpha.item() + 1e-6))


def evaluate_gentrification_model(x: Tensor, jobs_data=jobs_data_t0, building_permits_data=building_permits_data_t0, evaluate=False, tract_data=training_data) -> float:
    gravity_scores = pd.DataFrame()

    for tract in tract_data['GEOID']:
        lat, lon = census_tract_to_coordinates(tract)
        if pd.isna(lat) or pd.isna(lon):
            print(f"Skipping tract {tract} due to missing coordinates.")
            continue

        # determine gravity from jobs, weighted by distance
        job_gravity = 0
        jobs_data['distance'] = calculate_distances(lat, lon, jobs_data)
        job_gravity = calculate_gravity_score(jobs_data, distance=x[1], alpha=x[0])

        # determine gravity from building permits, weighted by distance
        building_permit_gravity = 0
        building_permits_data['distance'] = calculate_distances(lat, lon, building_permits_data)
        building_permit_gravity = calculate_gravity_score(building_permits_data, year_var='date', min_year=2008, distance=x[3], alpha=x[2])

        permit_counts = building_permits_data[building_permits_data['distance'] < x[3].item()].groupby('date').size()
        # use linear regression to find the slope of crime over time
        permit_model = LinearRegression()
        permit_model.fit(permit_counts.index.values.reshape(-1, 1), permit_counts.values)

        gravity_scores = pd.concat([gravity_scores, pd.DataFrame({
            'GEOID': tract, 'latitude': lat, 'longitude': lon,
            'job_gravity': job_gravity, 'building_permit_gravity': building_permit_gravity,
              'building_permit_trend': permit_model.coef_[0], 
              'gentrifying': tract_data[tract_data["GEOID"] == tract]["gentrifying"].values[0]}, index=[0])])
        
    # standardize all the gravity scores
    for column in gravity_scores.columns:
        if column in ['GEOID', 'latitude', 'longitude', 'gentrifying']:
            continue
        elif column in ['building_permit_trend']:
            # flip the signs in trend features for interpretability, so that an increasing building trend contributes negatively to gentrification and an increasing building permit trend contributes positively to gentrification
            gravity_scores[column] = -gravity_scores[column]
        gravity_scores[column] = (gravity_scores[column] - gravity_scores[column].mean()) / gravity_scores[column].std()

    # write to .csv file if applicable
    if evaluate:
        return gravity_scores
        # with open('t1_scores.csv', 'w') as f:
        #     f.write('GEOID,latitude,longitude,job_gravity,building_permit_gravity,building_permit_trend\n')
        #     f.write(f'{gravity_scores["GEOID"]},{gravity_scores["latitude"]},{gravity_scores["longitude"]},{gravity_scores["job_gravity"]},{gravity_scores["building_permit_gravity"]},{gravity_scores["building_permit_trend"]}\n'.format())  
        #     return 0.0
    else:
        feature_cols = [
            "job_gravity",
            "building_permit_gravity",
            "building_permit_trend", # signifies relative value to other tracts (higher value means better on building permit trend than the mean tract)
        ]

        X = gravity_scores[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        y = gravity_scores["gentrifying"].astype(int).values

        if len(np.unique(y)) < 2:
            return 0.0

        X_train, X_val, y_train, y_val = train_test_split(
            X,
            y,
            test_size=0.25,
            random_state=42,
            stratify=y
        )

        # Standardize using training data only
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = LogisticRegression(
            l1_ratio=0,
            C=1.0,
            max_iter=1000,
            class_weight="balanced"
        )

        model.fit(X_train_scaled, y_train)

        pred_probs = model.predict_proba(X_val_scaled)[:, 1]

        # Option A: maximize AUC
        #auc = roc_auc_score(y_val, pred_probs)

        # Option B: maximize negative log-loss instead
        neg_logloss = -log_loss(y_val, pred_probs)

        return float(neg_logloss)

Once we've determined which features actually matter, we can use Bayesian Optimization to determine the correct model weights, and then use this weighted model to predict which census tracts are at greatest risk of gentrification over the next 10 years.

## Model Pipeline
Parcel Data
Building Permit Data at t_0
LODES job Data at t_0

Hyperparameters:
permit_r - how far from the parcel to check for permits
permit_α - how much to weight nearby building permits
job_r - how far from the parcel to inspect jobs
job_α - how much to weight jobs



In [38]:
# import sys
# import botorch
# import os
# import time
# from contextlib import ExitStack

# import gpytorch
# import gpytorch.settings as gpts
# import torch
# from gpytorch.constraints import Interval
# from gpytorch.distributions import MultivariateNormal
# from gpytorch.kernels import RBFKernel, RFFKernel, ScaleKernel
# from gpytorch.likelihoods import GaussianLikelihood
# from gpytorch.mlls import ExactMarginalLogLikelihood
# from torch.quasirandom import SobolEngine
# from torch import Tensor

# from botorch.fit import fit_gpytorch_mll
# from botorch.generation import MaxPosteriorSampling
# from botorch.models import SingleTaskGP
# from botorch.test_functions import Hartmann
# from botorch.utils.sampling import draw_sobol_samples
# from botorch.sampling.pathwise.posterior_samplers import get_matheron_path_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
SMOKE_TEST = os.environ.get("SMOKE_TEST")

dim = 4
# x[0] = job_alpha
# x[1] = job_radius (in lat/long degrees)
# x[2] = permit_alpha
# x[3] = permit_radius (in lat/long degrees)
model_bounds = torch.tensor(
    [
        [0.5, 0.04, 0.5, 0.01], # lower bounds
        [3.0, 0.4, 3.0, 0.08], # upper bounds
    ],
    dtype=dtype,
    device=device,
)


def generate_batch(
    X: Tensor,
    Y: Tensor,
    batch_size: int,
    n_candidates: int,
    sampler: str,  # "cholesky", "ciq", "rff", "lanczos"
    seed: int,
) -> Tensor:
    assert sampler in ("cholesky", "ciq", "rff", "lanczos", "pathwise")
    assert X.min() >= 0.0 and X.max() < 3.0 and torch.all(torch.isfinite(Y))

    if sampler == "rff":
        base_kernel = RFFKernel(ard_num_dims=X.shape[-1], num_samples=1024)
    else:
        base_kernel = RBFKernel(ard_num_dims=X.shape[-1])
    covar_module = ScaleKernel(base_kernel)

    # Fit a GP model
    model = SingleTaskGP(train_X=X, train_Y=Y, covar_module=covar_module)
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll)

    # Draw samples on a Sobol sequence
    X_cand = draw_sobol_samples(bounds=model_bounds, n=n_candidates, q=1, seed=seed).squeeze(-2)

    # Thompson sample
    with ExitStack() as es:
        if sampler == "cholesky":
            es.enter_context(gpts.max_cholesky_size(float("inf")))
        elif sampler == "ciq":
            es.enter_context(gpts.fast_computations(covar_root_decomposition=True))
            es.enter_context(gpts.max_cholesky_size(0))
            es.enter_context(gpts.ciq_samples(True))
            es.enter_context(
                gpts.minres_tolerance(2e-3)
            )  # Controls accuracy and runtime
            es.enter_context(gpts.num_contour_quadrature(15))
        elif sampler == "lanczos":
            es.enter_context(
                gpts.fast_computations(
                    covar_root_decomposition=True, log_prob=True, solves=True
                )
            )
            es.enter_context(gpts.max_lanczos_quadrature_iterations(10))
            es.enter_context(gpts.max_cholesky_size(0))
            es.enter_context(gpts.ciq_samples(False))
        elif sampler == "rff":
            es.enter_context(gpts.fast_computations(covar_root_decomposition=True))
        elif sampler == "pathwise":
            model = get_matheron_path_model(model=model)
        es.enter_context(torch.no_grad())

        thompson_sampling = MaxPosteriorSampling(model=model, replacement=False)
        X_next = thompson_sampling(X_cand, num_samples=batch_size)

    return X_next

def run_optimization(
    sampler: str,
    n_candidates: int,
    n_init: int,
    max_evals: int,
    batch_size: int,
    seed: int,
) -> tuple[Tensor, Tensor]:
    X = draw_sobol_samples(bounds=model_bounds, n=n_init, q=1, seed=seed).squeeze(-2)
    Y = torch.tensor(
        [evaluate_gentrification_model(x) for x in X], dtype=dtype, device=device
    ).unsqueeze(-1)
    print(f"{len(X)}) Best value: {Y.max().item():.2e}")

    inner_seed = seed
    while len(X) < max_evals:
        # Create a batch
        start = time.monotonic()
        inner_seed += 1
        X_next = generate_batch(
            X=X,
            Y=Y,
            batch_size=min(batch_size, max_evals - len(X)),
            n_candidates=n_candidates,
            seed=inner_seed,
            sampler=sampler,
        )
        end = time.monotonic()
        print(f"Generated batch in {end - start:.1f} seconds")
        Y_next = torch.tensor(
            [evaluate_gentrification_model(x) for x in X_next], dtype=dtype, device=device
        ).unsqueeze(-1)

        # Append data
        X = torch.cat((X, X_next), dim=0)
        Y = torch.cat((Y, Y_next), dim=0)

        print(f"{len(X)}) Best value: {Y.max().item()} at parameter {X[Y.argmax()]}")
        print(f"{len(X)}) Worst value: {Y.min().item()} at parameter {X[Y.argmin()]}")
        print(f"{len(X)}) Batch outputs: {Y}")
    
    return X[Y.argmax()], Y.max()

batch_size = 5
n_init = 10
max_evals = 50
seed = 12345  # To get the same Sobol points
N_CAND = 10_000 if not SMOKE_TEST else 10

shared_args = {
    "n_candidates": N_CAND,
    "n_init": n_init,
    "max_evals": max_evals,
    "batch_size": batch_size,
    "seed": seed,
}

X_best, Y_best = run_optimization("cholesky", **shared_args)

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


From our weights that we found, plug them into the model for t1 and get our predictions.

In [10]:
# remove tracts defined as gentrified in t0
test_data = training_data[training_data['gentrifying'] == 0]

t0_scores = evaluate_gentrification_model(X_best, jobs_data=jobs_data_t0, building_permits_data=building_permits_data_t0, evaluate=True)
t1_scores = evaluate_gentrification_model(X_best, jobs_data=jobs_data_t1, building_permits_data=building_permits_data_t1, evaluate=True, tract_data=test_data)

feature_cols = [
    "job_gravity",
    "building_permit_gravity",
    "building_permit_trend",
]

# 1. Historical training data using X_best gravity parameters
train_df = t0_scores.copy()

X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = train_df["gentrifying"].astype(int)

# 2. Fit scaler on TRAINING data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 3. Train final predictive model
final_model = LogisticRegression(
    l1_ratio=0,
    C=1.0,
    max_iter=1000,
    class_weight="balanced"
)

final_model.fit(X_train_scaled, y_train)


# t1 gravity scores computed using X_best parameters
pred_df = t1_scores.copy()

X_pred = pred_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# Important: use the SAME scaler fitted on training data
X_pred_scaled = scaler.transform(X_pred)

# Predicted probability of future gentrification
pred_df["predicted_gentrification_prob"] = final_model.predict_proba(X_pred_scaled)[:, 1]

# Rank tracts
ranked = pred_df.sort_values("predicted_gentrification_prob", ascending=False)

ranked[["GEOID", "predicted_gentrification_prob"] + feature_cols].head(20)

ranked.to_csv('predictions_top20.csv', index=False)